# 面试问题：任务队列怎样实现 at-least-once、幂等、重试、DLQ，并避免重复副作用？

**一句话回答**：队列用 visibility lease 交付消息，worker 完成业务事务后才 ack；lease 到期会重投，因此处理器必须以 idempotency key 在权威存储中原子记录结果。可重试错误使用有上限的指数退避与 jitter，永久错误直接 DLQ；ack token 带 delivery generation，旧 worker 不能确认新租约。

本 Notebook 用确定性内存状态机模拟 worker 崩溃、重复交付、过期 receipt、幂等账本、重试和死信。

In [ ]:
from dataclasses import dataclass, replace  # 导入本单元所需的依赖。
from collections import Counter  # 导入本单元所需的依赖。
import hashlib, json, math  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

SEED86=8601; rng86=np.random.default_rng(SEED86)  # 计算并保存当前步骤的中间状态。
assert SEED86==8601  # 用受控断言验证关键不变量。
assert hashlib.sha256(b"job-1").hexdigest()!=hashlib.sha256(b"job-2").hexdigest()  # 用受控断言验证关键不变量。
assert np.isfinite(rng86.random())  # 用受控断言验证关键不变量。

## 1. Job、delivery 与业务幂等键是三种 ID

`job_id` 标识逻辑任务，`delivery_id/generation` 标识一次租约，`idempotency_key` 标识不可重复的业务副作用。消息重发可以换 delivery，但业务键不变。payload schema/version、tenant、创建时间和最大尝试次数也必须绑定。

幂等键若由可变 payload 某部分随意拼接，会把同一操作当不同任务；若过于宽泛，又会误吞合法新操作。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Job86:  # 定义承载本节状态与行为的数据结构。
    job_id:str; idempotency_key:str; tenant:str; payload:tuple; created_at:int; max_attempts:int=4  # 计算并保存当前步骤的中间状态。
    def __post_init__(self):  # 定义本节可复用的核心函数。
        if not self.job_id or not self.idempotency_key or not self.tenant or self.created_at<0 or self.max_attempts<1: raise ValueError("job_contract")  # 按当前条件选择后续控制路径。
@dataclass  # 为下方定义附加声明式配置。
class Message86:  # 定义承载本节状态与行为的数据结构。
    job:Job86; attempt:int=0; visible_at:int=0; lease_until:int|None=None; generation:int=0; acked:bool=False; last_error:str|None=None  # 计算并保存当前步骤的中间状态。
job86=Job86("j1","charge:order-7","t1",(("amount",100),),0)  # 计算并保存当前步骤的中间状态。
assert job86.max_attempts==4 and dict(job86.payload)["amount"]==100  # 用受控断言验证关键不变量。
try: Job86("","k","t",(),0); raise AssertionError("invalid job accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="job_contract"  # 捕获预期异常并验证失败分支。
assert Message86(job86).visible_at==0  # 用受控断言验证关键不变量。

## 2. visibility lease 与 generation receipt

claim 选择 `visible_at <= now` 且未 ack 的消息，增加 attempt/generation 并设置 lease。worker 必须在 lease 内 ack；超时后其他 worker 可重新 claim。receipt 包含 job ID 和 generation，旧 worker 完成得再晚也不能 ack 新 delivery。

visibility timeout 应覆盖常见处理时间，并允许心跳续租；无限续租要受最大墙钟时间约束。

In [ ]:
class Queue86:  # 定义承载本节状态与行为的数据结构。
    def __init__(self): self.messages={}; self.dlq={}; self.metrics=Counter()  # 定义本节可复用的核心函数。
    def enqueue(self,job,visible_at=None):  # 定义本节可复用的核心函数。
        if job.job_id in self.messages: return "duplicate_job_id"  # 按当前条件选择后续控制路径。
        self.messages[job.job_id]=Message86(job,visible_at=job.created_at if visible_at is None else visible_at); self.metrics["enqueued"]+=1; return "enqueued"  # 计算并保存当前步骤的中间状态。
    def claim(self,now,lease=5):  # 定义本节可复用的核心函数。
        choices=[m for m in self.messages.values() if not m.acked and m.visible_at<=now and (m.lease_until is None or m.lease_until<=now)]  # 计算并保存当前步骤的中间状态。
        if not choices: return None  # 按当前条件选择后续控制路径。
        m=min(choices,key=lambda x:(x.visible_at,x.job.job_id)); m.attempt+=1; m.generation+=1; m.lease_until=now+lease; self.metrics["delivered"]+=1  # 计算并保存当前步骤的中间状态。
        return m.job,(m.job.job_id,m.generation),m.attempt  # 返回当前分支计算出的结果。
    def ack(self,receipt,now):  # 定义本节可复用的核心函数。
        jid,generation=receipt; m=self.messages.get(jid)  # 计算并保存当前步骤的中间状态。
        if not m or m.acked or m.generation!=generation or m.lease_until is None or now>m.lease_until: return False  # 按当前条件选择后续控制路径。
        m.acked=True; self.metrics["acked"]+=1; return True  # 计算并保存当前步骤的中间状态。
q86=Queue86(); assert q86.enqueue(job86)=="enqueued" and q86.enqueue(job86)=="duplicate_job_id"  # 计算并保存当前步骤的中间状态。
first86=q86.claim(0,3); assert first86[2]==1 and q86.claim(1) is None  # 计算并保存当前步骤的中间状态。
second86=q86.claim(3,5); assert second86[2]==2 and second86[1]!=first86[1]  # 计算并保存当前步骤的中间状态。
assert not q86.ack(first86[1],3) and q86.ack(second86[1],4)  # 用受控断言验证关键不变量。

## 3. 长任务用受限 heartbeat 续租

worker 在 lease 到期前用当前 receipt 续租；generation 不匹配、已过期或已 ack 都拒绝。续租不能无限延长，可设置从首次 claim 起的最大墙钟时间，超限后取消/分片/DLQ，防止僵尸 worker 永久占有任务。

heartbeat 只延长可见性，不代表业务进度已持久化；长任务仍应保存 checkpoint。

In [ ]:
def heartbeat86(queue,receipt,now,extend_by,max_until):  # 定义本节可复用的核心函数。
    jid,generation=receipt; m=queue.messages.get(jid)  # 计算并保存当前步骤的中间状态。
    if not m or m.acked or m.generation!=generation or m.lease_until is None or now>=m.lease_until: return False  # 按当前条件选择后续控制路径。
    m.lease_until=min(max_until,m.lease_until+extend_by); return m.lease_until>now  # 计算并保存当前步骤的中间状态。
heartbeat_q86=Queue86(); heartbeat_q86.enqueue(Job86("jh","op:h","t1",(("amount",1),),0)); hd86=heartbeat_q86.claim(0,4)  # 计算并保存当前步骤的中间状态。
assert heartbeat86(heartbeat_q86,hd86[1],2,3,8) and heartbeat_q86.messages["jh"].lease_until==7  # 用受控断言验证关键不变量。
assert not heartbeat86(heartbeat_q86,("jh",99),3,3,8)  # 用受控断言验证关键不变量。
assert heartbeat86(heartbeat_q86,hd86[1],6,5,8) and heartbeat_q86.messages["jh"].lease_until==8  # 用受控断言验证关键不变量。

## 4. 幂等账本把 at-least-once 变成一次业务效果

worker 在同一数据库事务中：检查 idempotency key；若已有成功结果则直接返回；否则执行业务写并记录结果。不能先调用外部支付再写账本，否则两者之间崩溃仍会重复扣款；外部系统也需要同一幂等键或 outbox/saga。

下例账本保存 payload digest，防止相同 key 被不同金额复用。

In [ ]:
class Ledger86:  # 定义承载本节状态与行为的数据结构。
    def __init__(self): self.results={}; self.balance=0; self.effects=0  # 定义本节可复用的核心函数。
    def execute(self,job):  # 定义本节可复用的核心函数。
        payload_raw=json.dumps(dict(job.payload),sort_keys=True,separators=(",",":")); digest=hashlib.sha256(payload_raw.encode()).hexdigest(); old=self.results.get(job.idempotency_key)  # 计算并保存当前步骤的中间状态。
        if old:  # 按当前条件选择后续控制路径。
            if old[0]!=digest: raise RuntimeError("idempotency_payload_conflict")  # 按当前条件选择后续控制路径。
            return old[1],"replayed"  # 返回当前分支计算出的结果。
        amount=dict(job.payload).get("amount")  # 计算并保存当前步骤的中间状态。
        if not isinstance(amount,int) or amount<=0: raise ValueError("permanent_payload_error")  # 按当前条件选择后续控制路径。
        result=f"receipt-{len(self.results)+1}"; self.balance+=amount; self.effects+=1; self.results[job.idempotency_key]=(digest,result); return result,"applied"  # 计算并保存当前步骤的中间状态。
ledger86=Ledger86(); result1_86=ledger86.execute(job86); result2_86=ledger86.execute(job86)  # 计算并保存当前步骤的中间状态。
assert result1_86[0]==result2_86[0] and result2_86[1]=="replayed"  # 用受控断言验证关键不变量。
assert ledger86.balance==100 and ledger86.effects==1  # 用受控断言验证关键不变量。
try: ledger86.execute(Job86("j2",job86.idempotency_key,"t1",(("amount",200),),1)); raise AssertionError("conflict accepted")  # 尝试执行可能失败的受控操作。
except RuntimeError as e: assert str(e)=="idempotency_payload_conflict"  # 捕获预期异常并验证失败分支。

## 5. 指数退避、确定性 jitter 与 DLQ

transient error（超时、限流）可重试；validation/权限错误通常永久失败。退避 `min(cap, base*2^(attempt-1))` 加 jitter，避免大量任务同时恢复形成 thundering herd。超过 max attempts 后进入 DLQ，保留原因、最后 payload digest 和 trace。

retry 调度必须释放当前 lease，并设置下一次 visible_at；不能一边持有 lease 一边 sleep。

In [ ]:
def backoff86(job_id,attempt,base=2,cap=30):  # 定义本节可复用的核心函数。
    if attempt<1: raise ValueError("attempt_contract")  # 按当前条件选择后续控制路径。
    raw=min(cap,base*(2**(attempt-1))); h=int(hashlib.sha256(f"{job_id}:{attempt}".encode()).hexdigest()[:8],16); return raw*(.75+.5*(h/0xffffffff))  # 计算并保存当前步骤的中间状态。
def fail86(queue,receipt,now,error,retryable=True):  # 定义本节可复用的核心函数。
    jid,generation=receipt; m=queue.messages[jid]  # 计算并保存当前步骤的中间状态。
    if m.generation!=generation: return "stale_receipt"  # 按当前条件选择后续控制路径。
    m.last_error=error; m.lease_until=None  # 计算并保存当前步骤的中间状态。
    if not retryable or m.attempt>=m.job.max_attempts:  # 按当前条件选择后续控制路径。
        queue.dlq[jid]=m; m.acked=True; queue.metrics["dlq"]+=1; return "dlq"  # 计算并保存当前步骤的中间状态。
    m.visible_at=math.ceil(now+backoff86(jid,m.attempt)); queue.metrics["retry"]+=1; return "retry"  # 计算并保存当前步骤的中间状态。
retry_job86=Job86("jr","op:r","t1",(("amount",1),),0,3); qr86=Queue86(); qr86.enqueue(retry_job86)  # 计算并保存当前步骤的中间状态。
d1_86=qr86.claim(0,2); assert fail86(qr86,d1_86[1],1,"timeout")=="retry" and qr86.claim(1) is None  # 计算并保存当前步骤的中间状态。
assert backoff86("jr",2)>backoff86("jr",1) and backoff86("jr",10)<=37.5  # 用受控断言验证关键不变量。
assert qr86.messages["jr"].visible_at>1  # 用受控断言验证关键不变量。

## 6. DLQ 不是垃圾桶

DLQ 需要 owner、保留期、告警、可检索原因和受控 replay。修复代码后 replay 应生成新 delivery，但沿用原 idempotency key；若业务已经成功而 ack 丢失，账本会返回 replayed。永久非法消息不应自动循环回主队列。

下面跑满尝试次数，并验证只出现一条 DLQ 记录。

In [ ]:
now86=qr86.messages["jr"].visible_at  # 计算并保存当前步骤的中间状态。
d2_86=qr86.claim(now86,2); assert fail86(qr86,d2_86[1],now86+1,"timeout")=="retry"  # 计算并保存当前步骤的中间状态。
now86=qr86.messages["jr"].visible_at; d3_86=qr86.claim(now86,2)  # 计算并保存当前步骤的中间状态。
assert d3_86[2]==3 and fail86(qr86,d3_86[1],now86+1,"timeout")=="dlq"  # 用受控断言验证关键不变量。
assert list(qr86.dlq)==["jr"] and qr86.claim(now86+100) is None  # 用受控断言验证关键不变量。
assert qr86.metrics["retry"]==2 and qr86.metrics["dlq"]==1  # 用受控断言验证关键不变量。

## 7. 顺序、并发与热点 key

全局顺序会严重限制吞吐，通常只保证同一 entity/order key 串行。将 ordering key hash 到 partition，每 partition 维护 sequence；失败消息若阻塞后续，需要业务选择严格顺序或旁路 DLQ。幂等不等于可交换，`set status` 和 `increment` 的重放语义不同。

这里验证相同 key 稳定落到同一 partition，节点数变化会迁移，因而 partition count 也是版本合同。

In [ ]:
def partition86(ordering_key,partitions):  # 定义本节可复用的核心函数。
    if partitions<1 or not ordering_key: raise ValueError("partition_contract")  # 按当前条件选择后续控制路径。
    return int(hashlib.blake2b(ordering_key.encode(),digest_size=8).hexdigest(),16)%partitions  # 返回当前分支计算出的结果。
assert partition86("order-7",16)==partition86("order-7",16)  # 用受控断言验证关键不变量。
keys86=[f"order-{i}" for i in range(200)]; counts86=Counter(partition86(k,8) for k in keys86)  # 计算并保存当前步骤的中间状态。
assert set(counts86)==set(range(8)) and max(counts86.values())/min(counts86.values())<2.5  # 用受控断言验证关键不变量。
assert any(partition86(k,8)!=partition86(k,9) for k in keys86)  # 用受控断言验证关键不变量。

## 8. Producer 侧也需要 outbox

业务数据库写成功但 enqueue 失败会漏任务；先 enqueue 后事务回滚会产生幽灵任务。transactional outbox 在业务事务中写待发送行，relay 至少一次投递，consumer 用上述幂等账本吸收重复。这是“exactly-once effect”的组合协议，不是队列单方面保证。

观测要关联 outbox ID、job ID、delivery generation 和业务 idempotency key，但 payload 中的隐私应脱敏。

In [ ]:
manifest86={"schema":1,"delivery":"at_least_once","lease_seconds":30,"backoff":"bounded_exponential_jitter_v1","max_attempts":4,"idempotency_ledger":"authoritative_db","dlq_owner":"ml-platform"}  # 计算并保存当前步骤的中间状态。
raw86=json.dumps(manifest86,sort_keys=True,separators=(",",":")); digest86=hashlib.sha256(raw86.encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert len(digest86)==64 and manifest86["max_attempts"]==job86.max_attempts  # 用受控断言验证关键不变量。
assert q86.metrics["delivered"]==2 and q86.metrics["acked"]==1  # 用受控断言验证关键不变量。
assert ledger86.effects==1 and len(ledger86.results)==1  # 用受控断言验证关键不变量。
print({"deliveries":q86.metrics["delivered"],"effects":ledger86.effects,"dlq":len(qr86.dlq),"sha":digest86[:12]})  # 执行当前语句以推进本节示例。

## 9. 面试收束、参考与练习

回答闭环：三类 ID → visibility lease/generation → 事务内幂等账本 → 分类错误与退避 → DLQ 运维 → ordering partition → producer outbox → trace/发布。不要声称消息队列能凭一个开关提供端到端 exactly-once。

练习：实现 lease heartbeat；模拟 worker 在副作用后、ack 前崩溃；加入 tenant 并发配额；设计 DLQ replay 审批和审计。

参考：[Transactional Outbox](https://microservices.io/patterns/data/transactional-outbox.html)、[Amazon Builders' Library：幂等 API](https://aws.amazon.com/builders-library/making-retries-safe-with-idempotent-APIs/)、[Kafka exactly-once 设计](https://kafka.apache.org/documentation/#semantics)。